# Comparaison des dynamiques Swendsen-Wang (100% CuPy Optimisé)

Ce notebook implémente et compare trois dynamiques de clusters pour la détection de communautés sur un graphe signé (modèle de Sankararaman-Baccelli) comme décrit dans le Chapitre 11 de la thèse.

Les trois dynamiques sont :
- **SW-edges classique** : dynamique arête par arête.
- **SW-triangles (blancs)** : l'énergie des arêtes est transférée uniquement aux triangles orientés vers le haut.
- **SW-triangles (half-half)** : l'énergie de chaque arête est divisée en deux et partagée entre ses deux triangles adjacents (blanc et noir).

In [ ]:
!pip install cupy-cuda12x # A ajuster selon l'environnement si ce n'est pas sur Colab
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
from cupyx.scipy.sparse import csr_matrix
from cupyx.scipy.sparse.csgraph import connected_components
from tqdm.notebook import tqdm

# Paramètres
d = 2
p = 0.65
n = 10000
L = int(np.round(n ** (1/d)))
T = 1000
N_samples = 10

In [ ]:
def generate_graph(L):
    N = L * L
    # Génération de grille entièrement vectorisée directement sur GPU (aucun python loop !)
    y, x = cp.meshgrid(cp.arange(L), cp.arange(L), indexing='ij')
    idx = y * L + x
    
    h_target = y * L + (x + 1) % L
    h_edges = cp.stack((idx, h_target), axis=-1).reshape(-1, 2)
    
    v_target = ((y + 1) % L) * L + x
    v_edges = cp.stack((idx, v_target), axis=-1).reshape(-1, 2)
    
    d_target = ((y + 1) % L) * L + (x + 1) % L
    d_edges = cp.stack((idx, d_target), axis=-1).reshape(-1, 2)
    
    edges = cp.concatenate((h_edges, v_edges, d_edges), axis=0)
    
    v_edge_white = N + y * L + (x + 1) % L
    d_edge_white = 2 * N + idx
    white_triangles = cp.stack((idx, v_edge_white, d_edge_white), axis=-1).reshape(-1, 3)
    
    v_edge_black = N + idx
    h_edge_black = ((y + 1) % L) * L + x
    d_edge_black = 2 * N + idx
    black_triangles = cp.stack((v_edge_black, h_edge_black, d_edge_black), axis=-1).reshape(-1, 3)
    
    return edges, white_triangles, black_triangles

In [ ]:
def SW_step(sigma, W, edges, mode, up, triangles, type_A, au, self_loops):
    N_edges = len(edges)
    satisfied = (W * sigma[edges[:,0]] * sigma[edges[:,1]]) > 0
    
    if mode == 'edges':
        freeze_prob = float(1 - cp.exp(-up))
        frozen = satisfied & (cp.random.rand(N_edges) < freeze_prob)
    else:
        sat_tri = satisfied[triangles]
        sat_count = cp.sum(sat_tri, axis=1)
        
        rand_vals = cp.random.rand(len(triangles))
        freeze_A = type_A & (sat_count == 3) & (rand_vals < au)
        freeze_B = (~type_A) & (sat_count == 2) & (rand_vals < au)
        
        frozen_int = cp.zeros(N_edges, dtype=cp.int32)
        if cp.any(freeze_A):
            cp.add.at(frozen_int, triangles.flatten(), cp.repeat(freeze_A, 3).astype(cp.int32))
            
        # Freeze B: Choix vectorisé sans synchronisation CPU-GPU ni indexation booléenne
        S0 = sat_tri[:, 0]
        S2 = sat_tri[:, 2]
        C1 = 1 - S0.astype(cp.int32)
        C2 = 1 + S2.astype(cp.int32)
        
        pick = cp.random.randint(0, 2, size=len(triangles))
        chosen_col = cp.where(pick == 0, C1, C2)
        chosen_edges = triangles[cp.arange(len(triangles)), chosen_col]
        
        cp.add.at(frozen_int, chosen_edges, freeze_B.astype(cp.int32))
        frozen = frozen_int > 0

    # 100% GPU - CSR Matrix
    row = edges[frozen, 0]
    col = edges[frozen, 1]
    N_nodes = len(sigma)
    
    sym_row = cp.concatenate((row, col, self_loops))
    sym_col = cp.concatenate((col, row, self_loops))
    data = cp.ones(len(sym_row), dtype=cp.float32)
    
    adj = csr_matrix((data, (sym_row, sym_col)), shape=(N_nodes, N_nodes))
    n_comp, labels = connected_components(adj, directed=True)
    
    lcc_frac = cp.max(cp.bincount(labels)) / N_nodes
    
    # Flip optimisé GPU sans transfert de n_comp vers le CPU
    flip = cp.random.randint(0, 2, size=N_nodes) * 2 - 1
    sigma = sigma * flip[labels]
    
    return sigma, lcc_frac

In [ ]:
edges_gpu, white_tri_gpu, black_tri_gpu = generate_graph(L)
up = cp.log(p / (1 - p))
modes = ['edges', 'white', 'half-half']

overlap_history = {m: cp.zeros((N_samples, T)) for m in modes}
lcc_history = {m: cp.zeros((N_samples, T)) for m in modes}

for sample in tqdm(range(N_samples), desc="Samples"):
    Sigma = cp.random.randint(0, 2, size=L*L) * 2 - 1
    
    same_comm = Sigma[edges_gpu[:,0]] == Sigma[edges_gpu[:,1]]
    correct_obs = cp.random.rand(len(edges_gpu)) < p
    W = cp.where(same_comm == correct_obs, up, -up)
    
    for mode in modes:
        sigma = cp.random.randint(0, 2, size=L*L) * 2 - 1
        
        # Précalculs statiques hors de la boucle temporelle (évite les allocations GPU répétées !)
        if mode == 'edges':
            triangles = None
            type_A = None
            au = 0.0
        elif mode == 'white':
            triangles = white_tri_gpu
            type_A = cp.prod(W[triangles], axis=1) > 0
            au = float(1 - cp.exp(-2 * up))
        else:
            triangles = cp.vstack((white_tri_gpu, black_tri_gpu))
            type_A = cp.prod(W[triangles], axis=1) > 0
            au = float(1 - cp.exp(-up))
            
        self_loops = cp.arange(L*L, dtype=cp.int32)
        
        for t in range(T):
            sigma, lcc = SW_step(sigma, W, edges_gpu, mode, up, triangles, type_A, au, self_loops)
            ov = cp.abs(cp.mean(sigma * Sigma))
            
            overlap_history[mode][sample, t] = ov
            lcc_history[mode][sample, t] = lcc

In [ ]:
mean_ov_gpu = {m: cp.mean(overlap_history[m], axis=0).get() for m in modes}
std_ov_gpu = {m: cp.std(overlap_history[m], axis=0).get() for m in modes}
mean_lcc_gpu = {m: cp.mean(lcc_history[m], axis=0).get() for m in modes}
std_lcc_gpu = {m: cp.std(lcc_history[m], axis=0).get() for m in modes}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

colors = {'edges': 'blue', 'white': 'orange', 'half-half': 'green'}
labels = {'edges': 'SW-Edges', 'white': 'SW-Triangles (Blancs)', 'half-half': 'SW-Triangles (Half-Half)'}

for mode in modes:
    ax1.plot(mean_ov_gpu[mode], label=labels[mode], color=colors[mode])
    ax1.fill_between(range(T), mean_ov_gpu[mode] - std_ov_gpu[mode], mean_ov_gpu[mode] + std_ov_gpu[mode], color=colors[mode], alpha=0.2)
    
    ax2.plot(mean_lcc_gpu[mode], label=labels[mode], color=colors[mode])
    ax2.fill_between(range(T), mean_lcc_gpu[mode] - std_lcc_gpu[mode], mean_lcc_gpu[mode] + std_lcc_gpu[mode], color=colors[mode], alpha=0.2)

ax1.set_title('Overlap (Recouvrement) vs Itérations')
ax1.set_xlabel('Itération')
ax1.set_ylabel('Overlap moyen')
ax1.legend()
ax1.grid(True)

ax2.set_title('Taille de la plus grande composante (LCC) vs Itérations')
ax2.set_xlabel('Itération')
ax2.set_ylabel('Proportion de points dans la LCC')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()